# Optimisation bayésienne avec Optuna : plus malin que la grille

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/cercle-des-experts/blob/main/ml-classique-et-deep-learning/tuning-hyperparametres/tuning-hyperparametres.ipynb)

📖 **Le mini-cours complet** : [Le tuning d'hyperparamètres (mais sans migraine)](https://www.monshotdata.com/p/tuning-hyperparametres)

🧭 **Le cours** : [ML classique et Deep Learning › 2.2 Régression et classification](https://www.monshotdata.com/mini-cours/ml-classique-et-deep-learning#h2sc22)

---

Chercher les meilleurs hyperparamètres d'un modèle en testant toutes les combinaisons, c'est fiable mais très coûteux. La recherche aléatoire est plus économe mais joue aux dés. L'**optimisation bayésienne** apprend de chaque essai pour choisir le suivant. Tu vas comparer les trois méthodes sur le même Random Forest, avec le même budget de validation croisée.

**À la fin de ce notebook, tu sauras :**

- comment lancer une recherche par grille (`GridSearchCV`) et une recherche aléatoire (`RandomizedSearchCV`) ;
- comment écrire une fonction objectif et une étude **Optuna** pour un Random Forest ;
- comment comparer les méthodes honnêtement : nombre d'essais, temps de calcul, score, et pourquoi il faut fixer les graines aléatoires.


### Avant de lancer

Aucun GPU. Compte **10 à 20 minutes** sur les 2 cœurs de Colab, dont l'essentiel pour la recherche par grille (720 combinaisons × 5 plis) : lance-la et va prendre un café. Les temps affichés dépendent de ta machine et du nombre de cœurs ; seuls leurs ordres de grandeur comptent. **Correction par rapport à la version diffusée avec le mini-cours :** le Random Forest de la fonction objectif Optuna n'avait pas de graine aléatoire (`random_state`), alors que celui de la grille en avait une. Les scores n'étaient donc pas strictement comparables ; tout est maintenant fixé à `42`.

🧪 *Testé le 20 novembre 2026 avec Python 3.13, scikit-learn 1.9 et Optuna 5.1.0.*

## Étape 1. Installer Optuna

Optuna n'est pas installé par défaut sur Colab.

In [1]:
!pip install -q optuna

## Étape 2. Importations

On importe Optuna (avec son échantillonneur bayésien **TPE**), les outils de recherche de scikit-learn et un Random Forest.

In [2]:
import optuna
import optuna.logging

from sklearn import datasets
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from optuna.samplers import TPESampler
from time import time

optuna.logging.disable_default_handler()

## Étape 3. Charger les données

Le jeu *digits* de scikit-learn : 1 797 images 8×8 de chiffres manuscrits, aplaties en 64 variables. 10 classes.

In [3]:
digits = datasets.load_digits()

n_samples = len(digits.images)
X = digits.images.reshape((n_samples, -1))
y = digits.target

## Étape 4. Le classifieur

Un Random Forest avec une graine fixée pour que les résultats soient reproductibles.

In [4]:
# Objet classificateur Random Forest
rfc = RandomForestClassifier(random_state=42)

## Étape 5. L'espace de recherche

Les six hyperparamètres à régler. La grille contient 4 × 2 × 3 × 5 × 2 × 3 = **720 combinaisons**. Chacune est évaluée par validation croisée à 5 plis, soit 3 600 entraînements pour la grille complète.

In [5]:
hyperparam_grid = {
    'n_estimators': [50,100,150,200],
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 3, 4],
    'min_samples_leaf': [1, 2, 3, 4, 5],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [5, 6, 7]
    }

## Étape 6. Une fonction de résumé

Pour comparer les méthodes sur les mêmes critères : temps total, nombre d'essais, numéro de l'essai où le meilleur score est trouvé, meilleur score et meilleurs hyperparamètres.

In [6]:
def get_best_results(param_object, total_time):

    iterations = len(param_object.cv_results_['params'])
    best_score = param_object.best_score_
    best_index = param_object.best_index_+1
    best_params = param_object.best_params_

    print(f"---{param_object.__class__.__name__}---")
    print(f"Temps total : {total_time:.2f} secondes")
    print(f"Nombre d'itérations :{iterations}")
    print(f"Meilleur essai (index) :{best_index}")
    print(f"Meilleur score :{best_score}")
    print(f"Meilleurs hyperparamètres :{best_params}")

## Étape 7. Les deux méthodes classiques

### 7.1 Recherche par grille

On teste **toutes** les combinaisons : exhaustif, donc le meilleur score possible dans cet espace, mais le plus coûteux.

In [7]:
grid_search = GridSearchCV(estimator=rfc,
                  param_grid=hyperparam_grid,
                  scoring='f1_micro',
                  n_jobs=-1,
                  verbose=0)

start = time()
grid_search.fit(X, y)
total_time = time() - start

get_best_results(grid_search, total_time)

---GridSearchCV---
Temps total : 1223.89 secondes
Nombre d'itérations :720
Meilleur essai (index) :667
Meilleur score :0.9365769111730116
Meilleurs hyperparamètres :{'criterion': 'entropy', 'max_depth': 7, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 150}


### 7.2 Recherche aléatoire

On tire 360 combinaisons au hasard, soit la moitié de la grille. Plus rapide, mais rien ne garantit de tomber sur la meilleure.

In [8]:
random_search = RandomizedSearchCV(estimator=rfc,
                  param_distributions=hyperparam_grid,
                  scoring='f1_micro',
                  n_jobs=-1,
                  verbose=0,
                  n_iter=360,
                  random_state=42)

# effectuer l'optimisation des hyperparamètres
start = time()
random_search.fit(X, y)
total_time = time() - start

# stocker les résultats dans un dataframe
get_best_results(random_search, total_time)

---RandomizedSearchCV---
Temps total : 297.91 secondes
Nombre d'itérations :360
Meilleur essai (index) :184
Meilleur score :0.9360244506344785
Meilleurs hyperparamètres :{'n_estimators': 200, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 7, 'criterion': 'entropy'}


## Étape 8. L'optimisation bayésienne avec Optuna

Trois étapes : une **fonction objectif** (qui entraîne un modèle pour un jeu d'hyperparamètres et renvoie son score), une **étude** (qui pilote la recherche), puis `optimize`. Après chaque essai, l'algorithme TPE met à jour sa croyance sur les zones prometteuses de l'espace et y concentre les essais suivants.

La fonction objectif reprend exactement le même espace de recherche, le même modèle (même `random_state`) et la même validation croisée que la grille : c'est ce qui rend la comparaison honnête.

In [9]:
N_TRIALS = 100

def objective(trial):

    # espace de recherche (le même que celui de la grille)
    n_estimators = trial.suggest_int('n_estimators', low=50, high=200, step=50)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    min_samples_split = trial.suggest_int('min_samples_split', low=2, high=4, step=1)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', low=1, high=5, step=1)
    max_depth = trial.suggest_int('max_depth', low=5, high=7, step=1)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])

    # classificateur Random Forest (même graine que pour la grille)
    rfc = RandomForestClassifier(n_estimators=n_estimators,
                                 criterion=criterion,
                                 min_samples_split=min_samples_split,
                                 min_samples_leaf=min_samples_leaf,
                                 max_depth=max_depth,
                                 max_features=max_features,
                                 random_state=42)

    # retourner le score (validation croisée à 5 plis)
    return cross_val_score(estimator=rfc,
                           X=X,
                           y=y,
                           scoring='f1_micro',
                           n_jobs=-1).mean()


# créer une étude (objectif : maximiser le score)
study = optuna.create_study(sampler=TPESampler(seed=42), direction='maximize')

# effectuer l'optimisation des hyperparamètres (en chronométrant le processus)
start = time()
study.optimize(objective, n_trials=N_TRIALS)
total_time = time() - start

print("---Optimisation Bayésienne---")
print(f"Temps total : {total_time:.2f} secondes")
print(f"Nombre d'itérations : {N_TRIALS}")
print(f"Meilleur essai (index) : {study.best_trial.number + 1}")
print(f"Meilleur score : {study.best_trial.value}")
print(f"Meilleurs hyperparamètres : {study.best_params}")

---Optimisation Bayésienne---
Temps total : 127.17 secondes
Nombre d'itérations : 100
Meilleur essai (index) : 50
Meilleur score : 0.9365769111730116
Meilleurs hyperparamètres : {'n_estimators': 150, 'criterion': 'entropy', 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_depth': 7, 'max_features': 'log2'}


## Ce qu'il faut retenir

Dans notre exécution (graines fixées à 42) :

| Méthode | Essais | Meilleur score F1 |
|---|---|---|
| Grille | 720 | 0,9366 |
| Aléatoire | 360 | 0,9360 |
| Optuna (bayésien) | 100 | 0,9366 |

- Optuna retrouve **exactement la meilleure combinaison de la grille**, avec 7 fois moins d'essais, et la trouve dès l'essai n°50 ; c'est aussi la méthode la plus rapide.
- La recherche aléatoire passe tout près de l'optimum avec la moitié des essais, mais sans garantie.
- Dans un espace de recherche **discret et fini** comme celui-ci, la grille est exhaustive : aucune méthode ne peut la dépasser. Si tu vois un autre score plus élevé qu'elle, vérifie que les comparaisons sont faites à graine égale (un Random Forest sans `random_state` varie légèrement d'un entraînement à l'autre).
- L'avantage de l'optimisation bayésienne grandit avec le **coût d'un essai** et la **taille de l'espace** (hyperparamètres continus, réseaux de neurones…) : c'est là que la grille devient inutilisable.

---

## Pour aller plus loin

- [Arbres de décision : L'erreur que tout le monde fait (et comment l'éviter)](https://www.monshotdata.com/p/surapprentissage-arbres-decision) — pourquoi les hyperparamètres d'un arbre comptent
- [Calibration des modèles ML (Partie 1)](https://www.monshotdata.com/p/calibration-modeles-machine-learning) — des probabilités fiables, après le réglage